In [2]:

# STEP 1: DATA ACQUISITION

import pandas as pd
from pathlib import Path

# Dataset Path (Change this to your folder)
DATA_PATH = Path(
    "/Users/dishitasingh/Desktop/Drexel/INFO442/INFO442-project-main/INFO442-project/Kaggle_dataset"
)

# Rating Files
rating_files = [
    DATA_PATH / "combined_data_1.txt",
    DATA_PATH / "combined_data_2.txt",
    DATA_PATH / "combined_data_3.txt",
    DATA_PATH / "combined_data_4.txt"
]

print("Checking rating files...\n")

for file in rating_files:
    if file.exists():
        size_gb = file.stat().st_size / (1024**3)
        print(f"✓ {file.name} ({size_gb:.2f} GB)")
    else:
        print(f"✗ {file.name} NOT FOUND")


# Load Movie Titles
movie_file = DATA_PATH / "movie_titles.csv"
movie_records = []
with open(movie_file, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        movie_id, year, title = line.split(",", 2)

        movie_records.append({
            "movie_id": int(movie_id),
            "year": pd.to_numeric(year, errors="coerce"),
            "title": title
        })
movies = pd.DataFrame(movie_records)
movies["year"] = movies["year"].astype("Int64")


# Display Movie Information
print("\nMovie Titles Loaded Successfully!\n")
print(movies.head())

print("\nMovie Dataset Shape:")
print(movies.shape)


Checking rating files...

✓ combined_data_1.txt (0.46 GB)
✓ combined_data_2.txt (0.52 GB)
✓ combined_data_3.txt (0.43 GB)
✓ combined_data_4.txt (0.51 GB)

Movie Titles Loaded Successfully!

   movie_id  year                         title
0         1  2003               Dinosaur Planet
1         2  2004    Isle of Man TT 2004 Review
2         3  1997                     Character
3         4  1994  Paula Abdul's Get Up & Dance
4         5  2004      The Rise and Fall of ECW

Movie Dataset Shape:
(17770, 3)


In [3]:
# STEP 2: Parse and Merge the Four Rating Files
# Output: ratings.csv

output_file = DATA_PATH / "ratings.csv"

# Create output file with header
with open(output_file, "w") as out:
    out.write("user_id,movie_id,rating,date\n")

for file in rating_files:
    print(f"Processing {file.name}...")
    current_movie = None
    rows = []

    with open(file, "r") as f:
        for line in f:
            line = line.strip()

            # Movie ID line
            if line.endswith(":"):
                current_movie = int(line[:-1])

            # Rating line
            else:
                user_id, rating, date = line.split(",")
                rows.append([
                    int(user_id),
                    current_movie,
                    int(rating),
                    date
                ])

            # Save every 1 million rows
            if len(rows) >= 1_000_000:
                pd.DataFrame(
                    rows,
                    columns=["user_id", "movie_id", "rating", "date"]
                ).to_csv(
                    output_file,
                    mode="a",
                    header=False,
                    index=False
                )
                print(f"  Saved {len(rows):,} rows")
                rows = []

    # Save remaining rows
    if rows:
        pd.DataFrame(
            rows,
            columns=["user_id", "movie_id", "rating", "date"]
        ).to_csv(
            output_file,
            mode="a",
            header=False,
            index=False
        )
        print(f"  Saved final {len(rows):,} rows")

print("\nAll rating files merged successfully!")
print(f"Saved to: {output_file}")


Processing combined_data_1.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved final 53,764 rows
Processing combined_data_2.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1

In [ ]:
# Load and clean the movie titles table first (handling commas in titles)
movie_records = []
with open(DATA_PATH / "movie_titles.csv", "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Split only on the first two commas to keep commas inside titles intact
        movie_id, year, title = line.split(",", 2)
        movie_records.append({
            "movie_id": int(movie_id),
            "year": pd.to_numeric(year, errors="coerce"),
            "title": title
        })

movies = pd.DataFrame(movie_records)

In [5]:
movies_clean = movies.dropna(subset=['year'])
valid_movie_ids = set(movies_clean['movie_id'])

# 2. Define memory-efficient types
dtypes = {
    'user_id': 'int32',
    'movie_id': 'int32',
    'rating': 'int8'
}

# 3. Read ONLY the columns we need
ratings = pd.read_csv(output_file, dtype=dtypes, usecols=['user_id', 'movie_id', 'rating'])

# 4. Filter the ratings using our clean movie IDs
ratings = ratings[ratings['movie_id'].isin(valid_movie_ids)]

print("Ratings loaded and cleaned successfully.")
print(ratings.shape)

Ratings loaded and cleaned successfully.
(100479542, 3)


In [ ]:
# STEP 3: Clean the Data

# 1. Convert data types
ratings["user_id"] = pd.to_numeric(
    ratings["user_id"],
    errors="coerce"
)
ratings["movie_id"] = pd.to_numeric(
    ratings["movie_id"],
    errors="coerce"
)
ratings["rating"] = pd.to_numeric(
    ratings["rating"],
    errors="coerce"
)

# Convert numeric columns to integers after removing nulls
ratings["user_id"] = ratings["user_id"].astype("int32")
ratings["movie_id"] = ratings["movie_id"].astype("int32")
ratings["rating"] = ratings["rating"].astype("int8")

# 4. Validate rating range
invalid_ratings = ~ratings["rating"].isin([1, 2, 3, 4, 5])
print(
    f"\nRows with invalid ratings: "
    f"{invalid_ratings.sum():,}"
)
# Remove invalid ratings if any exist
ratings = ratings[
    ratings["rating"].isin([1, 2, 3, 4, 5])
]

# 5. Validate movie ID range
invalid_movie_ids = ~ratings["movie_id"].between(
    1,
    17770
)
print(
    f"Rows with invalid movie IDs: "
    f"{invalid_movie_ids.sum():,}"
)
# Remove invalid movie IDs if any exist
ratings = ratings[
    ratings["movie_id"].between(1, 17770)
]

# 6. Final verification
print("\nFinal data types:")
print(ratings.dtypes)
print("\nMissing values after cleaning:")
print(ratings.isnull().sum())
print("\nRating values:")
print(sorted(ratings["rating"].unique()))
print(
    "\nMovie ID range:",
    ratings["movie_id"].min(),
    "to",
    ratings["movie_id"].max()
)
print(f"\nFinal number of rows: {len(ratings):,}")
print("\nCleaned data sample:")
print(ratings.head())


Rows with invalid ratings: 0
Rows with invalid movie IDs: 0

Final data types:
user_id     int32
movie_id    int32
rating       int8
dtype: object

Missing values after cleaning:
user_id     0
movie_id    0
rating      0
dtype: int64

Rating values:
[np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5)]

Movie ID range: 1 to 17770

Final number of rows: 100,479,542

Cleaned data sample:
   user_id  movie_id  rating
0  1488844         1       3
1   822109         1       5
2   885013         1       4
3    30878         1       4
4   823519         1       3


In [11]:
# STEP 6: Filter Sparse Data

TOP_USERS = 10000
MIN_MOVIE_RATINGS = 50

# Keep the 10,000 most active users
user_counts = ratings.groupby("user_id").size()

top_users = user_counts.nlargest(TOP_USERS).index

ratings_filtered = ratings[
    ratings["user_id"].isin(top_users)
].copy()

# Keep movies with at least 50 ratings
movie_counts = ratings_filtered.groupby("movie_id").size()

popular_movies = movie_counts[
    movie_counts >= MIN_MOVIE_RATINGS
].index

ratings_filtered = ratings_filtered[
    ratings_filtered["movie_id"].isin(popular_movies)
].copy()

# Summary
print("=" * 50)
print("FILTERED DATASET SUMMARY")
print("=" * 50)

print(f"Original Ratings : {len(ratings):,}")
print(f"Filtered Ratings : {len(ratings_filtered):,}")

print(f"\nOriginal Users : {ratings['user_id'].nunique():,}")
print(f"Filtered Users : {ratings_filtered['user_id'].nunique():,}")

print(f"\nOriginal Movies : {ratings['movie_id'].nunique():,}")
print(f"Filtered Movies : {ratings_filtered['movie_id'].nunique():,}")

print(f"\nData Retained: {(len(ratings_filtered)/len(ratings))*100:.2f}%")

FILTERED DATASET SUMMARY
Original Ratings : 100,479,542
Filtered Ratings : 15,351,759

Original Users : 480,189
Filtered Users : 10,000

Original Movies : 17,763
Filtered Movies : 13,503

Data Retained: 15.28%


In [12]:

# STEP 7: Build the User–Item Sparse Matrix

from scipy.sparse import csr_matrix

# Create consecutive matrix positions for user IDs and movie IDs
user_ids = ratings_filtered["user_id"].unique()
movie_ids = ratings_filtered["movie_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}
movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}
index_to_user = {
    index: user_id
    for user_id, index in user_to_index.items()
}
index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

# Convert IDs into row and column positions
user_indices = ratings_filtered["user_id"].map(user_to_index)
movie_indices = ratings_filtered["movie_id"].map(movie_to_index)

# Build sparse user-item matrix
# Rows = users
# Columns = movies
# Values = ratings
user_item_matrix = csr_matrix(
    (
        ratings_filtered["rating"].astype("float32"),
        (user_indices, movie_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

# Verify the matrix
print("User–Item Matrix Created Successfully")
print(f"Matrix shape       : {user_item_matrix.shape}")
print(f"Number of users    : {user_item_matrix.shape[0]:,}")
print(f"Number of movies   : {user_item_matrix.shape[1]:,}")
print(f"Stored ratings     : {user_item_matrix.nnz:,}")

total_cells = (
    user_item_matrix.shape[0]
    * user_item_matrix.shape[1]
)

density = (
    user_item_matrix.nnz
    / total_cells
) * 100

sparsity = 100 - density

print(f"Matrix density     : {density:.4f}%")
print(f"Matrix sparsity    : {sparsity:.4f}%")

User–Item Matrix Created Successfully
Matrix shape       : (10000, 13503)
Number of users    : 10,000
Number of movies   : 13,503
Stored ratings     : 15,351,759
Matrix density     : 11.3691%
Matrix sparsity    : 88.6309%


In [13]:
#item-user sparce matrix

# Rows = movies
# Columns = users
item_user_matrix = user_item_matrix.T.tocsr()

print("\nItem–User Matrix Shape:")
print(item_user_matrix.shape)


Item–User Matrix Shape:
(13503, 10000)


In [ ]:
# STEP 9: Apply Truncated SVD (linear dimentionality reduction)
# applied to user–item matrix

from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 100

svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=42
)

movie_features = svd.fit_transform(item_user_matrix)

print("Truncated SVD Completed")
print(f"Original item-user shape : {item_user_matrix.shape}")
print(f"Reduced movie shape      : {movie_features.shape}")

explained_variance = svd.explained_variance_ratio_.sum()

print(f"Number of components     : {N_COMPONENTS}")
print(f"Explained variance       : {explained_variance:.2%}")

Truncated SVD Completed
Original item-user shape : (13503, 10000)
Reduced movie shape      : (13503, 100)
Number of components     : 100
Explained variance       : 57.70%
